# Sampling Activations from MouseNet

Extracts layer activations from **MouseNetCompletePool** (Shi et al., PLOS CompBio 2022) — a biologically-constrained CNN of the mouse visual cortex based on the Allen connectome.

No pip install needed — the repo lives at `../Mouse_CNN/` and is added to `sys.path` directly.

**Pretrained ImageNet weights:** `../Mouse_CNN/cmouse/exps/imagenet/1111_model_best.pth.tar`

**Input size:** 64×64 RGB

**Areas available (hooked at BatchNorm output):**  
`LGNd`, `VISp4`, `VISp2/3`, `VISp5`, `VISl4`, `VISl2/3`, `VISl5`, `VISrl4`, `VISrl2/3`, `VISrl5`,  
`VISli4`, `VISli2/3`, `VISli5`, `VISpl4`, `VISpl2/3`, `VISpl5`, `VISal4`, `VISal2/3`, `VISal5`, `VISpor4`, `VISpor2/3`, `VISpor5`

Output: `data/sampled/tensor4d_mousenet_<AREA>_i3_n<N>_seed17.npy` (shape: neurons × 11 × 8 × 37).

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import sys, os, pickle
_d = os.path.abspath(os.getcwd())
sys.path.insert(0, _d if os.path.isdir(os.path.join(_d, 'src')) else os.path.dirname(_d))
from src.plot_utils import createFlowDataset
from time import time

# MouseNet: no package install — add repo dirs to path directly
MOUSE_CNN_ROOT = os.path.abspath('../Mouse_CNN')
assert os.path.isdir(MOUSE_CNN_ROOT), f'Mouse_CNN not found at {MOUSE_CNN_ROOT}'
sys.path.insert(0, MOUSE_CNN_ROOT)
sys.path.insert(0, os.path.join(MOUSE_CNN_ROOT, 'cmouse'))

# Mock Allen Brain / mcmodels dependencies — only needed to build anatomy from scratch;
# we load from a pre-built pickle so these are never called at runtime.
from unittest.mock import MagicMock
for _mod in ['mcmodels', 'mcmodels.core', 'mcmodels.core.VoxelModelCache']:
    sys.modules.setdefault(_mod, MagicMock())

from mousenet_complete_pool import MouseNetCompletePool
import network as _network_mod  # triggers anatomy import (needs mocks above)

def load_network_from_pickle(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

######################## PARAMS ########################

# Pretrained weights and network architecture pickle (ImageNet)
NET_PKL = os.path.join(MOUSE_CNN_ROOT, 'cmouse/exps/imagenet/myresults/network_complete_updated_number(3,64,64).pkl')
WEIGHTS = os.path.join(MOUSE_CNN_ROOT, 'cmouse/exps/imagenet/1111_model_best.pth.tar')

# Areas to sample — hooks are placed on BNs[area] (post-BatchNorm features)
# Edit this list to add/remove areas. Area names with '/' are valid dict keys.
AREAS_TO_SAMPLE = [
    'LGNv',
    'VISp4', 'VISp2/3', 'VISp5',
    'VISl2/3', 'VISal2/3', 'VISrl2/3', 'VISli2/3', 'VISpl2/3',
]

n_fmaps_to_sample = 25
samples_per_fmap  = 40
seed              = 17
N_INSTANCES       = 3
trial_len         = 37
NDIRS             = 8
scl_factor        = 0.7
BATCH_SIZE        = 64
MOUSENET_SIZE     = 64   # MouseNet expects 64×64 input

# Optical-flow stimulus parameters (same as nb01)
topdir      = '../stimuli/flowstims'
orig_shape  = (800, 600)
input_shape = (144, 256)
mydirs      = ['0', '45', '90', '135', '180', '225', '270', '315']
categories  = [
    'grat_W12', 'grat_W1', 'grat_W2',
    'neg1dotflow_D1_bg', 'neg1dotflow_D2_bg',
    'neg3dotflow_D1_bg', 'neg3dotflow_D2_bg',
    'pos1dotflow_D1_bg', 'pos1dotflow_D2_bg',
    'pos3dotflow_D1_bg', 'pos3dotflow_D2_bg',
]
NSTIMS = len(categories)
assert NSTIMS == 11

n_orig_imgs    = NSTIMS * NDIRS    # 88
n_shifted_imgs = n_orig_imgs * trial_len  # 3256

IMGNET_MEAN = np.array([0.485, 0.456, 0.406], dtype='float32')
IMGNET_STD  = np.array([0.229, 0.224, 0.225], dtype='float32')

print(f'n_orig_imgs={n_orig_imgs}, n_shifted_imgs={n_shifted_imgs}')

Using device: cpu
n_orig_imgs=88, n_shifted_imgs=3256


In [2]:
########## LOAD MODEL + PRETRAINED WEIGHTS ##########

# config.py currently has INPUT_SIZE=(3,100,100) but the pretrained pickle
# was built for (3,64,64) — patch before constructing the model.
import example.config as _cfg
_cfg.INPUT_SIZE = (3, MOUSENET_SIZE, MOUSENET_SIZE)

# Load network architecture from pickle
net = load_network_from_pickle(NET_PKL)
print('Network architecture loaded.')

# Instantiate model
model = MouseNetCompletePool(net, mask=3)
model.eval()

# Load pretrained ImageNet weights
# weights_only=False needed because the checkpoint contains numpy arrays (legacy format)
checkpoint = torch.load(WEIGHTS, map_location='cpu', weights_only=False)
model.load_state_dict(checkpoint['state_dict'])
print(f'Weights loaded. Best ImageNet acc@1: {checkpoint["best_acc1"]:.2f}%  (epoch {checkpoint["epoch"]})')

model = model.to(device)

# Print available BN areas
print('\nAvailable BN areas:', list(model.BNs.keys()))

Network architecture loaded.
{'inputLGNv': (0, 100, 0, 100), 'LGNvVISp4': (0, 42, 12, 30), 'VISp4VISp2/3': (0, 85, 0, 60), 'VISp4VISl4': (3, 43, 20, 33), 'VISp4VISrl4': (7, 46, 2, 35), 'VISp4VISli4': (26, 20, 20, 35), 'VISp4VISpl4': (26, 59, 20, 37), 'VISp4VISal4': (17, 29, 20, 20), 'VISp4VISpor4': (26, 20, 20, 9), 'VISp2/3VISp5': (0, 85, 0, 60), 'VISp2/3VISl4': (3, 43, 20, 33), 'VISp2/3VISrl4': (7, 46, 2, 35), 'VISp2/3VISli4': (26, 20, 20, 35), 'VISp2/3VISpl4': (26, 59, 20, 37), 'VISp2/3VISal4': (17, 29, 20, 20), 'VISp2/3VISpor4': (26, 20, 20, 9), 'VISl4VISl2/3': (0, 21, 0, 16), 'VISl4VISpor4': (11, 10, 0, 4), 'VISrl4VISrl2/3': (0, 23, 0, 17), 'VISrl4VISpor4': (9, 10, 9, 4), 'VISli4VISli2/3': (0, 10, 0, 17), 'VISli4VISpor4': (0, 10, 0, 4), 'VISpl4VISpl2/3': (0, 29, 0, 18), 'VISpl4VISpor4': (0, 10, 0, 4), 'VISal4VISal2/3': (0, 14, 0, 10), 'VISal4VISpor4': (4, 10, 0, 4), 'VISpor4VISpor2/3': (0, 10, 0, 4), 'VISp5VISl4': (3, 43, 20, 33), 'VISp5VISrl4': (7, 46, 2, 35), 'VISp5VISli4': (26, 

In [3]:
########## REGISTER HOOKS ON BN LAYERS ##########
# We hook on model.BNs[area] (BatchNorm2d) to capture post-BN spatial feature maps.

activation_outputs = {}

def make_hook(area_name):
    def hook(module, inp, out):
        activation_outputs[area_name] = out.detach().cpu()
    return hook

for area in AREAS_TO_SAMPLE:
    assert area in model.BNs, f'Area "{area}" not found in model.BNs. Available: {list(model.BNs.keys())}'
    model.BNs[area].register_forward_hook(make_hook(area))

# Warm-up pass to determine output shapes
dummy = torch.zeros(1, 3, MOUSENET_SIZE, MOUSENET_SIZE, device=device)
with torch.no_grad():
    _ = model(dummy)

area_shapes = {}
for area in AREAS_TO_SAMPLE:
    out = activation_outputs[area]   # (1, C, H, W)
    _, C, H, W = out.shape
    area_shapes[area] = (C, H, W)
    print(f'  {area:15s}: C={C:4d}, H={H:3d}, W={W:3d}  → {C*H*W:,} units')

  LGNv           : C=   5, H= 64, W= 64  → 20,480 units
  VISp4          : C=  26, H= 64, W= 64  → 106,496 units
  VISp2/3        : C=  42, H= 64, W= 64  → 172,032 units
  VISp5          : C=  32, H= 64, W= 64  → 131,072 units
  VISl2/3        : C=  21, H= 32, W= 32  → 21,504 units
  VISal2/3       : C=  15, H= 32, W= 32  → 15,360 units
  VISrl2/3       : C=  22, H= 32, W= 32  → 22,528 units
  VISli2/3       : C=   9, H= 32, W= 32  → 9,216 units
  VISpl2/3       : C=  17, H= 32, W= 32  → 17,408 units


In [4]:
########## LOAD OPTICAL-FLOW STIMULI ##########

flow_datasets = createFlowDataset(
    categories, topdir, mydirs,
    orig_shape=orig_shape, input_shape=input_shape,
    scl_factor=scl_factor, N_INSTANCES=N_INSTANCES,
    trial_len=trial_len, stride=1,
)
assert flow_datasets[0].shape[0] == n_shifted_imgs
print('Stimuli loaded.')

*INSTANCE 0 ...........
*INSTANCE 1 ...........
*INSTANCE 2 ...........
Stimuli loaded.


In [5]:
########## PREPROCESSING HELPER ##########

def preprocess_frames(flat_batch, orig_h, orig_w, target_size):
    """Ravel grayscale uint8 frames → normalised RGB tensor."""
    B = flat_batch.shape[0]
    frames = flat_batch.reshape(B, orig_h, orig_w).astype('float32') / 255.0
    frames_rgb = np.stack([frames, frames, frames], axis=1)  # (B, 3, H, W)
    frames_norm = (frames_rgb - IMGNET_MEAN[:, None, None]) / IMGNET_STD[:, None, None]
    t = torch.tensor(frames_norm)
    t = F.interpolate(t, size=(target_size, target_size), mode='bilinear', align_corners=False)
    return t

In [6]:
########## FORWARD PASSES (all areas captured simultaneously) ##########

orig_h, orig_w = input_shape  # 144, 256

layer_outputs = {area: np.zeros((n_shifted_imgs, *area_shapes[area]), dtype='float32')
                 for area in AREAS_TO_SAMPLE}

for insti in range(N_INSTANCES):
    extX = flow_datasets[insti]   # (n_shifted_imgs, H*W)
    print(f'Instance {insti}', flush=True)
    t0 = time()

    for start in range(0, n_shifted_imgs, BATCH_SIZE):
        print(start)
        batch_flat = extX[start:start + BATCH_SIZE]
        batch_t = preprocess_frames(batch_flat, orig_h, orig_w, MOUSENET_SIZE).to(device)
        with torch.no_grad():
            _ = model(batch_t)
        bs = batch_t.shape[0]
        for area in AREAS_TO_SAMPLE:
            layer_outputs[area][start:start + bs] += activation_outputs[area].numpy()

    print(f'  done in {time()-t0:.1f}s')

for area in AREAS_TO_SAMPLE:
    layer_outputs[area] /= N_INSTANCES
    lo = layer_outputs[area]
    print(f'{area:15s}: {lo.shape}  min={lo.min():.3f}  max={lo.max():.3f}')

Instance 0
0
64
128
192
256
320
384
448
512
576
640
704
768
832
896
960
1024
1088
1152
1216
1280
1344
1408
1472
1536
1600
1664
1728
1792
1856
1920
1984
2048
2112
2176
2240
2304
2368
2432
2496
2560
2624
2688
2752
2816
2880
2944
3008
3072
3136
3200
  done in 450.9s
Instance 1
0
64
128
192
256
320
384
448
512
576
640
704
768
832
896
960
1024
1088
1152
1216
1280
1344
1408
1472
1536
1600
1664
1728
1792
1856
1920
1984
2048
2112
2176
2240
2304
2368
2432
2496
2560
2624
2688
2752
2816
2880
2944
3008
3072
3136
3200
  done in 497.4s
Instance 2
0
64
128
192
256
320
384
448
512
576
640
704
768
832
896
960
1024
1088
1152
1216
1280
1344
1408
1472
1536
1600
1664
1728
1792
1856
1920
1984
2048
2112
2176
2240
2304
2368
2432
2496
2560
2624
2688
2752
2816
2880
2944
3008
3072
3136
3200
  done in 555.8s
LGNv           : (3256, 5, 64, 64)  min=0.000  max=11.104
VISp4          : (3256, 26, 64, 64)  min=0.000  max=17.061
VISp2/3        : (3256, 42, 64, 64)  min=0.000  max=12.375
VISp5          : (3256, 32, 64, 

In [7]:
########## SAMPLE NEURONS + BUILD TENSOR4D + SAVE (per area) ##########

os.makedirs('../data/sampled', exist_ok=True)

for area in AREAS_TO_SAMPLE:
    print(f'\n=== {area} ===')
    C, H, W = area_shapes[area]
    lo = layer_outputs[area]   # (n_shifted_imgs, C, H, W)
    n_neurons_per_fmap = H * W

    # Reshape to (n_orig_imgs, C, H*W, trial_len)
    lo_t = lo.reshape(n_orig_imgs, trial_len, C, H, W)
    lo_t = lo_t.transpose(0, 2, 3, 4, 1)             # (n_orig_imgs, C, H, W, trial_len)
    all_per_img_output = lo_t.reshape(n_orig_imgs, C, H * W, trial_len)  # (88, C, H*W, 37)

    all_neurons_maxs  = all_per_img_output.max(axis=(0, 3))   # (C, H*W)
    all_neurons_means = all_per_img_output.mean(axis=(0, 3))  # (C, H*W)

    # Sample feature maps (maxFr)
    np.random.seed(seed)
    maxsmean = all_neurons_maxs.mean(1)  # (C,)
    nonzero_fmaps = int((~np.isclose(maxsmean, 0)).sum())
    n_fmaps_ = min(n_fmaps_to_sample, nonzero_fmaps)
    probs_fmap = maxsmean / maxsmean.sum()
    top_fmaps = np.random.choice(C, n_fmaps_, replace=False, p=probs_fmap)

    # Sample neurons within each fmap (maxNr)
    samps_per_fmap = min(samples_per_fmap, n_neurons_per_fmap)
    sampled_neurons = []
    for fi in top_fmaps:
        neuron_vals = all_neurons_maxs[fi]
        nonzero_n = int((~np.isclose(neuron_vals, 0)).sum())
        samps = min(samps_per_fmap, nonzero_n)
        if samps == 0:
            continue
        probs_n = neuron_vals / neuron_vals.sum()
        top_nis = np.random.choice(n_neurons_per_fmap, samps, replace=False, p=probs_n)
        sampled_neurons += list(fi * n_neurons_per_fmap + top_nis)
    sampled_neurons = np.array(sampled_neurons)
    n_neurons_to_pick = len(sampled_neurons)
    print(f'  Sampled {n_neurons_to_pick} neurons from {n_fmaps_} feature maps')

    # Build tensor4d
    tensorX      = np.zeros((n_neurons_to_pick, NSTIMS, NDIRS, trial_len), dtype='float32')
    neurons_used = np.empty((n_neurons_to_pick, 3), dtype='int')  # (fmap_idx, i, j)

    for nii, ni in enumerate(sampled_neurons):
        fi   = ni // n_neurons_per_fmap
        posi = ni %  n_neurons_per_fmap
        ii   = posi // W
        jj   = posi %  W
        neurons_used[nii] = [fi, ii, jj]
        for cati in range(NSTIMS):
            pst = all_per_img_output[cati * NDIRS:(cati + 1) * NDIRS, fi, posi, :]
            tensorX[nii, cati] = pst

    print(f'  tensorX shape: {tensorX.shape}')

    # '/' in area names is invalid in filenames — replace with '_'
    area_safe = area.replace('/', '_')
    SUFFIX = f'mousenet_{area_safe}_i{N_INSTANCES}_n{n_neurons_to_pick}_seed{seed}'
    out_tensor  = f'../data/sampled/tensor4d_{SUFFIX}.npy'
    out_neurons = f'../data/sampled/neurons_used_{SUFFIX}.npy'

    if os.path.exists(out_tensor):
        print(f'  [SKIP] already exists — delete to regenerate')
    else:
        np.save(out_tensor, tensorX)
        np.save(out_neurons, neurons_used)
        print(f'  Saved {out_tensor}')

print('\n--- _LAYER_PARAMS entries for notebook 06 ---')
for area in AREAS_TO_SAMPLE:
    area_safe = area.replace('/', '_')
    # guess n from saved file
    import glob as _glob
    matches = _glob.glob(f'../data/sampled/tensor4d_mousenet_{area_safe}_*.npy')
    if matches:
        arr = np.load(matches[0], mmap_mode='r')
        n = arr.shape[0]
        sfx = f'mousenet_{area_safe}_i{N_INSTANCES}_n{n}_seed{seed}'
        print(f'  "{sfx}": dict(low_sf={"True" if "VISp" in area or "LGN" in area else "False"}),')


=== LGNv ===
  Sampled 200 neurons from 5 feature maps
  tensorX shape: (200, 11, 8, 37)
  Saved ../data/sampled/tensor4d_mousenet_LGNv_i3_n200_seed17.npy

=== VISp4 ===
  Sampled 1000 neurons from 25 feature maps
  tensorX shape: (1000, 11, 8, 37)
  Saved ../data/sampled/tensor4d_mousenet_VISp4_i3_n1000_seed17.npy

=== VISp2/3 ===
  Sampled 1000 neurons from 25 feature maps
  tensorX shape: (1000, 11, 8, 37)
  Saved ../data/sampled/tensor4d_mousenet_VISp2_3_i3_n1000_seed17.npy

=== VISp5 ===
  Sampled 1000 neurons from 25 feature maps
  tensorX shape: (1000, 11, 8, 37)
  Saved ../data/sampled/tensor4d_mousenet_VISp5_i3_n1000_seed17.npy

=== VISl2/3 ===
  Sampled 840 neurons from 21 feature maps
  tensorX shape: (840, 11, 8, 37)
  Saved ../data/sampled/tensor4d_mousenet_VISl2_3_i3_n840_seed17.npy

=== VISal2/3 ===
  Sampled 600 neurons from 15 feature maps
  tensorX shape: (600, 11, 8, 37)
  Saved ../data/sampled/tensor4d_mousenet_VISal2_3_i3_n600_seed17.npy

=== VISrl2/3 ===
  Sample